In [1]:
# =========================
# 1. Imports
# =========================
import pandas as pd
import numpy as np
import re
from collections import Counter

from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


In [2]:

# =========================
# 2. Load Data
# =========================
data = pd.read_csv("amazon_cells_labelled.txt", delimiter='\t', header=None)
data.columns = ['Sentence', 'Class']
data['Sentence'] = data['Sentence'].astype(str).str.strip()

texts = data['Sentence'].tolist()
labels = data['Class'].tolist()

# =========================
# 3. Split Data
# =========================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts, labels, test_size=0.2, random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, random_state=42
)


In [3]:

# =========================
# 4. Tokenization
# =========================
def tokenize(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9 ]', '', text)
    return text.split()

# =========================
# 5. Vocabulary
# =========================
counter = Counter()
for text in train_texts:
    counter.update(tokenize(text))

word2idx = {"<PAD>": 0, "<UNK>": 1}
for i, (word, _) in enumerate(counter.most_common(5000 - 2), start=2):
    word2idx[word] = i


In [4]:

# =========================
# 6. Encoding
# =========================
max_len = 30

def encode(text):
    tokens = tokenize(text)
    seq = [word2idx.get(word, 1) for word in tokens]

    if len(seq) < max_len:
        seq += [0] * (max_len - len(seq))
    else:
        seq = seq[:max_len]

    return seq


In [5]:

# =========================
# 7. Dataset
# =========================
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = [encode(t) for t in texts]
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.texts[idx], dtype=torch.long),
            torch.tensor(self.labels[idx], dtype=torch.float32)
        )


In [6]:

# =========================
# 8. DataLoaders
# =========================
batch_size = 16

train_loader = DataLoader(TextDataset(train_texts, train_labels), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TextDataset(val_texts, val_labels), batch_size=batch_size)
test_loader = DataLoader(TextDataset(test_texts, test_labels), batch_size=batch_size)


In [7]:

# =========================
# 9. Models
# =========================

# ---- LSTM ----
class LSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        out = torch.mean(lstm_out, dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out.squeeze()

# ---- Bi-LSTM ----
class BiLSTMModel(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)
        out = torch.mean(lstm_out, dim=1)
        out = self.dropout(out)
        out = self.fc(out)
        return out.squeeze()

# ---- Bi-LSTM + Attention (BEST) ----

class BiLSTM_Attention(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=96):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)

        self.lstm = nn.LSTM(
            embed_dim,
            hidden_dim,
            batch_first=True,
            bidirectional=True
        )

        # Simpler attention (more stable)
        self.attention = nn.Sequential(
            nn.Linear(hidden_dim * 2, 64),
            nn.Tanh(),
            nn.Linear(64, 1)
        )

        self.dropout = nn.Dropout(0.6)
        self.fc = nn.Linear(hidden_dim * 2, 1)

    def forward(self, x):
        x = self.embedding(x)
        lstm_out, _ = self.lstm(x)

        attn_weights = torch.softmax(self.attention(lstm_out), dim=1)
        context = torch.sum(attn_weights * lstm_out, dim=1)

        out = self.dropout(context)
        out = self.fc(out)

        return out.squeeze()


In [8]:

# =========================
# 10. Train Function
# =========================
def train_model(model, name, train_loader, val_loader, epochs=6):
    print(f"\n===== Training {name} =====")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)
    
    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for x, y in train_loader:
            x, y = x.to(device), y.to(device)

            optimizer.zero_grad()
            outputs = model(x)

            loss = criterion(outputs, y)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1} | Loss: {total_loss/len(train_loader):.4f}")

    print(f"\nValidation Accuracy for {name}:")
    evaluate(model, val_loader)

    return model


In [9]:

# =========================
# 11. Evaluation
# =========================
def evaluate(model, loader):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)

            outputs = model(x)
            preds = (torch.sigmoid(outputs) > 0.5).float()

            correct += (preds == y).sum().item()
            total += y.size(0)

    acc = (correct / total) * 100
    print(f"Accuracy: {acc:.2f}% ({correct}/{total})")
    return acc


In [11]:

# =========================
# 12. Train All Models
# =========================
lstm_model = train_model(LSTMModel(len(word2idx)), "LSTM", train_loader, val_loader)
bilstm_model = train_model(BiLSTMModel(len(word2idx)), "Bi-LSTM", train_loader, val_loader)
attention_model = train_model(BiLSTM_Attention(len(word2idx)), "Bi-LSTM + Attention", train_loader, val_loader)


# =========================
# 13. Final Test Results
# =========================
print("\n===== FINAL TEST RESULTS =====")

print("\nLSTM Test Accuracy:")
evaluate(lstm_model, test_loader)

print("\nBi-LSTM Test Accuracy:")
evaluate(bilstm_model, test_loader)

print("\nBi-LSTM + Attention Test Accuracy:")
evaluate(attention_model, test_loader)    


===== Training LSTM =====
Epoch 1 | Loss: 0.6915
Epoch 2 | Loss: 0.6569
Epoch 3 | Loss: 0.4533
Epoch 4 | Loss: 0.3355
Epoch 5 | Loss: 0.2202
Epoch 6 | Loss: 0.1620

Validation Accuracy for LSTM:
Accuracy: 79.00% (79/100)

===== Training Bi-LSTM =====
Epoch 1 | Loss: 0.6911
Epoch 2 | Loss: 0.6634
Epoch 3 | Loss: 0.5083
Epoch 4 | Loss: 0.3727
Epoch 5 | Loss: 0.2995
Epoch 6 | Loss: 0.2060

Validation Accuracy for Bi-LSTM:
Accuracy: 80.00% (80/100)

===== Training Bi-LSTM + Attention =====
Epoch 1 | Loss: 0.6920
Epoch 2 | Loss: 0.6743
Epoch 3 | Loss: 0.5366
Epoch 4 | Loss: 0.4114
Epoch 5 | Loss: 0.3093
Epoch 6 | Loss: 0.2208

Validation Accuracy for Bi-LSTM + Attention:
Accuracy: 79.00% (79/100)

===== FINAL TEST RESULTS =====

LSTM Test Accuracy:
Accuracy: 77.00% (77/100)

Bi-LSTM Test Accuracy:
Accuracy: 84.00% (84/100)

Bi-LSTM + Attention Test Accuracy:
Accuracy: 73.00% (73/100)


73.0